# Task 3 — GRPO training for Mini-Factorio

Trains a LoRA adapter on `Qwen2.5-Coder-1.5B-Instruct` using TRL's `GRPOTrainer` and our simulator-backed reward (delta of composite reward per edit sequence). Evaluates base vs trained checkpoints on the val split and produces the metrics table + reward-vs-iteration plot the plan calls for.

**Runs on:** Colab T4 (target). Local Mac only for the dry-run smoke test — full runs will be very slow on CPU/MPS.

**Reward shape:** `delta = composite(after_edit) - composite(before_edit)`. Parse fail → -1.0. Empty / all-invalid edit list → 0.0. This kills the `"output []"` reward hack.

**Iteration model (simplification):** the plan describes 3 outer iterations refreshing `π_ref` between them. This notebook uses one continuous training run with checkpoints saved at 3 intermediate step counts, evaluated as `policy_1`, `policy_2`, `policy_final`. `π_ref` stays frozen at the base model. This is not strict iterative GRPO but is the common practical form and captures the mean-reward-vs-iteration story the plan wants.

## 1. Setup

Uncomment the install cell on a fresh Colab. On the local venv the packages are already there via `uv sync --extra llm`.

In [ ]:
# Colab setup — run once per fresh session.
!pip install -q 'transformers>=4.45' 'trl>=0.11' 'peft>=0.13' 'accelerate>=0.34' 'bitsandbytes>=0.43' 'datasets>=2.20' 'pydantic>=2.7'

# Clone the repo (private — you'll be prompted for a GitHub token).
!git clone https://github.com/Andrii238/ml-project.git /content/ml_project
%cd /content/ml_project


In [ ]:
import sys, pathlib
# Make repo imports work when the notebook is opened in-place.
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram GB:', torch.cuda.get_device_properties(0).total_memory / 1e9)

## 2. Dry-run smoke test

Two steps, group size 2, four training layouts. Confirms:
- Trainer instantiates without version errors
- Reward function returns floats
- KL / loss log per step
- Adapter saves to disk

**Do this before any real training.** If this fails, no real run will succeed either.

In [ ]:
!python -m training.train_grpo --dry-run --output-dir ./ckpts/dry

## 3. Baseline eval (policy_0)

Before we spend any training compute, sanity-check the baseline. If `parse_ok_pct` is very low (<30%), GRPO will have almost nothing to reward — add a small SFT warmup with a handful of correct examples first. Otherwise proceed.

In [ ]:
from training.evaluate import evaluate_checkpoint
from training.data import SplitSizes, build_val_layouts

val_layouts = build_val_layouts(SplitSizes(train=60, val=20))
policy_0 = evaluate_checkpoint('policy_0', None, val_layouts, samples_per_layout=4)
print(policy_0)

## 4. Full GRPO training

Defaults: G=8, LR 5e-5, β=0.04, μ=1, 200 optimizer steps, checkpoint every 50 steps. Adjust `--max-steps` up if reward is still climbing at the end.

## 3a. (Optional) Skip SFT if adapter already trained locally

If you have an `ckpts/sft/` from a local run, upload the folder via Colab file browser (or mount Drive). Skip the SFT training cell below.


In [ ]:
# GRPO training starting from SFT adapter (policy_1 → policy_final).
# --init-adapter points at the SFT checkpoint from Stage 1 (train_sft.py).
!python -m training.train_grpo \
  --init-adapter ./ckpts/sft \
  --output-dir ./ckpts/grpo \
  --max-steps 200 \
  --save-steps 50


## 5. Evaluate checkpoints

`policy_0` = base model; `policy_1..policy_final` = adapters saved during training.

In [ ]:
from pathlib import Path
from training.evaluate import evaluate_all
from dataclasses import asdict
import json

ckpt_dir = Path('./ckpts/grpo')
ckpt_paths = sorted(p for p in ckpt_dir.glob('checkpoint-*') if p.is_dir(),
                    key=lambda p: int(p.name.split('-')[-1]))
# Map: policy_0 = raw base, policy_1 = SFT (Stage 1), policy_2..policy_final = GRPO checkpoints.
checkpoints = [('policy_0', None), ('policy_1', './ckpts/sft')]
for i, p in enumerate(ckpt_paths):
    name = f'policy_{i+2}' if i+2 < len(ckpt_paths) + 1 else 'policy_final'
    checkpoints.append((name, str(p)))
print('Evaluating:', checkpoints)

metrics = evaluate_all(checkpoints, samples_per_layout=4, n_val=20)
Path('results').mkdir(exist_ok=True)
with open('results/checkpoint_eval.json', 'w') as f:
    json.dump([asdict(m) for m in metrics], f, indent=2)
for m in metrics:
    print(m.name, 'composite=', m.mean_composite, 'gs=', m.mean_green_science)


## 6. Metrics table

Raw per-checkpoint metrics (plan §Reward reporting). Composite is what GRPO optimizes; the other columns are the substantive story.

In [ ]:
import pandas as pd
from dataclasses import asdict
df = pd.DataFrame([asdict(m) for m in metrics])
df = df[['name', 'mean_green_science', 'mean_materials', 'mean_cells', 'mean_machines', 'valid_output_pct', 'parse_ok_pct', 'mean_composite']]
df.round(3)

## 7. Reward vs iteration

In [ ]:
import matplotlib.pyplot as plt

names = [m.name for m in metrics]
composites = [m.mean_composite for m in metrics]
gs = [m.mean_green_science for m in metrics]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(names, composites, marker='o')
ax1.set_title('Mean composite reward vs checkpoint')
ax1.set_ylabel('composite reward')
ax1.grid(True, alpha=0.3)

ax2.plot(names, gs, marker='o', color='tab:green')
ax2.set_title('Mean green-science rate vs checkpoint')
ax2.set_ylabel('items/sec')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_vs_iteration.png', dpi=120)
plt.show()

## 8. Per-iteration deltas

Task 3 claim (plan): `Δ_final > max_i Δ_i` where `Δ_i = mean(policy_{i+1}) - mean(policy_i)`. Deltas here are consecutive differences of the composite curve above.

In [ ]:
deltas = [composites[i+1] - composites[i] for i in range(len(composites) - 1)]
delta_labels = [f'{names[i]}→{names[i+1]}' for i in range(len(composites) - 1)]

plt.figure(figsize=(8, 4))
plt.bar(delta_labels, deltas)
plt.axhline(0, color='k', lw=0.5)
plt.title('Per-checkpoint reward delta')
plt.ylabel('Δ composite reward')
plt.xticks(rotation=20)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('reward_deltas.png', dpi=120)
plt.show()

total = composites[-1] - composites[0]
max_step = max(deltas) if deltas else 0.0
print(f'Total gain (policy_0 → policy_final): {total:+.3f}')
print(f'Max single-step delta: {max_step:+.3f}')
print(f'Claim `total > max single-step`: {"PASS" if total > max_step else "FAIL"}')